# For part 1 and 2 refer Redis notebook

Perfect. 🚀

You've now completed the most important Redis concepts:

✅ What Redis is

✅ Why in-memory cache fails

✅ Distributed cache

✅ TTL

✅ Cache hits/misses

✅ Cache invalidation

✅ Versioned cache keys

✅ Shared cache vs shared user data

✅ Multi-server architecture


---

🚀 Day 15 (Part 3) — Dockerizing the Recommendation System

Now we're moving from:

Runs on my laptop

to

Runs inside a container

This is a huge step because recruiters/interviewers love seeing:

FastAPI
Redis
Docker

on a project.


---

First: What Is Docker Actually?

Right now:

Windows
 ├── Python
 ├── FastAPI
 ├── Redis Client
 ├── Pandas
 ├── Numpy
 └── Surprise

Everything is installed directly on your machine.


---

Problem:

Suppose you send project to me.

I may have:

Python 3.11

You have:

Python 3.12

Maybe:

surprise version differs

Maybe:

numpy version differs

Project breaks.


---

Docker says:

Don't send code only.

Send:
Code
+
Environment
+
Dependencies

inside a container.


---

Container Analogy

Think:

Container
=
Mini Computer

that contains:

Python
FastAPI
Pandas
Models
Code

and behaves exactly the same everywhere.


---

Step 1 — Create requirements.txt

First question:

Do you already have a requirements.txt file?

Check project root:

recommendation_system/
│
├── api/
├── src/
├── models/
├── requirements.txt  ← ?

If not, run:

pip freeze > requirements.txt

Then open it and tell me:

Does it contain hundreds of packages?

or

Only project-related packages?

This matters because we'll clean it before Dockerizing.


---

Why I'm Asking

Many beginners accidentally generate:

requirements.txt

with:

500 packages

because it includes everything installed on their laptop.

For Docker we want only what's actually needed.


---

Once you show me the status of requirements.txt, we'll create:

Dockerfile

which is the blueprint Docker uses to build your application image. 🐳

That's the next major milestone toward a fully containerized recommendation system.

Perfect 🔥

You now understand:

✅ FastAPI
✅ Hybrid Recommendation Engine
✅ Model Versioning
✅ Startup Events
✅ In-Memory Cache
✅ Redis Cache
✅ Cache Invalidation Strategies
✅ Multi-worker Problem
✅ Shared Cache Architecture

Now we move to the next major milestone:

Day 15 - Part 3: Dockerization 🐳


---

First, What Is Docker?

Before Docker:

Your Laptop
│
├── Python 3.11
├── FastAPI
├── Redis
├── Numpy
├── Pandas
└── Surprise

If I give this project to another developer:

Works on Mugundhan's Laptop ❌
Fails on another laptop ❌

because:

Different Python Version
Different Package Version
Missing Dependency


---

Docker solves this.

Think:

Project
+
Python
+
Packages
+
Configuration
=
Docker Image

Now anyone can run:

docker run my-app

and get exactly the same environment.


---

Docker Concepts

There are only 3 things to understand.

1. Image

Blueprint.

Like:

Class

in OOP.

Example:

Recommendation System Image


---

2. Container

Running instance.

Like:

Object

in OOP.

Example:

recommendation-api-container


---

3. Dockerfile

Recipe.

Example:

Use Python

Install packages

Copy code

Run FastAPI


---

Step 1: Create Dockerfile

In project root create:

Dockerfile

No extension.

Project structure:

recommendation_system
│
├── api
├── src
├── model
├── logs
├── requirements.txt
├── Dockerfile
└── ...


---

Step 2: Add Dockerfile Content

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8001

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8001"]


---

Understanding Every Line

Base Image

FROM python:3.11-slim

Meaning:

Start with lightweight Python 3.11


---

Working Directory

WORKDIR /app

Equivalent to:

cd app

inside container.


---

Copy Requirements

COPY requirements.txt .

Copies:

Host → Container


---

Install Packages

RUN pip install --no-cache-dir -r requirements.txt

This creates Docker layers.


---

Copy Project

COPY . .

Copies entire project.


---

Expose Port

EXPOSE 8001

Documents:

Container listens on 8001


---

Start FastAPI

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8001"]

When container starts:

FastAPI starts automatically


---

Step 3: Build Image

From project root:

docker build -t recommendation-api .

Notice the dot:

Current Folder

is build context.


---

Expected Result

Something like:

Successfully tagged recommendation-api:latest


---

Concept Check

Before running it, answer this:

When Docker executes:

COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .

Why do we install packages before copying the whole project?

Hint:

Think about what happens when you modify only:

api/app.py

and rebuild the image.

This is one of the most important Docker optimization concepts (layer caching). Once you answer, we'll run the image and then connect Redis using Docker Compose. 🚀

It is like setting up new windows laptop...initially we will setup the workspace then we can install any apps needed..like that we are setting up the environment for that project then we are importing the project...by the way what is layer caching?


Your laptop analogy is actually pretty good for understanding why we install dependencies before copying the code, but the deeper reason is Docker Layer Caching.


---

# What is a Docker Layer?

Every instruction in a Dockerfile creates a layer.

For example:

FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .

becomes:

Layer 1 -> Python Image
Layer 2 -> WORKDIR /app
Layer 3 -> requirements.txt copied
Layer 4 -> pip install executed
Layer 5 -> project files copied

Think of it like a game save system 🎮:

Save Point 1
Save Point 2
Save Point 3
Save Point 4
Save Point 5

Docker remembers every step.


---

First Build

Suppose you run:

docker build -t recommendation-api .

Docker does:

Step 1 -> Download Python Image
Step 2 -> Create /app
Step 3 -> Copy requirements.txt
Step 4 -> Install numpy, pandas, fastapi, surprise...
Step 5 -> Copy project code

This might take a few minutes.


---

Now You Change One Line

Suppose tomorrow you modify:

print("Hello")

inside:

api/app.py

and rebuild:

docker build -t recommendation-api .


---

What Docker Sees

Docker checks:

FROM python:3.11-slim

Same as before ✅

WORKDIR /app

Same as before ✅

requirements.txt

Same as before ✅

Therefore:

pip install

does NOT run again.

Docker reuses the cached layer.

Only:

COPY . .

runs again.

Build finishes very quickly.


---

Why This Order Matters

Imagine you wrote:

COPY . .
RUN pip install -r requirements.txt

instead.

Now every time you change:

app.py

Docker sees:

COPY . .

changed ❌

which means:

RUN pip install -r requirements.txt

must run again ❌

Every rebuild would reinstall:

```text
numpy
pandas
scikit-learn
surprise

which is very slow.


---

Real World Example

Imagine Netflix has:

100 developers

and each developer rebuilds containers:

20 times/day

Without layer caching:

pip install
pip install
pip install
pip install
...

all day long 😭

With layer caching:

Use existing dependency layer

Builds become much faster.


---

Quick Visualization

Good Dockerfile:

COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .

Code change
     ↓
Only last layer rebuilds

Bad Dockerfile:

COPY . .
RUN pip install -r requirements.txt

Code change
     ↓
Dependencies reinstall every build


---

Concept Check

Suppose today your image builds successfully.

Tomorrow you only change:

CACHE_TTL = 400

to

CACHE_TTL = 600

in app.py.

When you run:

docker build -t recommendation-api .

Will Docker:

1. Reinstall all packages again?


2. Reuse the package installation layer and only rebuild the code layer?



And why? This is the exact thought process Docker uses internally. 🚀

Exactly. 👍

When Docker builds an image, COPY does not execute your Python code.

Example:

FROM python:3.11

WORKDIR /app

COPY requirements.txt .

RUN pip install -r requirements.txt

COPY . .

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8001"]

Let's break it down.

Layer 1

FROM python:3.11

Docker downloads the base image once.


---

Layer 2

WORKDIR /app

Creates /app folder.


---

Layer 3

COPY requirements.txt .

Copies only the requirements file.

Nothing executes.


---

Layer 4

RUN pip install -r requirements.txt

This executes.

This is usually the slowest layer.


---

Layer 5

COPY . .

Copies all project files.

Still nothing executes.

Docker simply places:

api/
src/
model/
logs/
requirements.txt

inside the image.

No Python code runs.

No model loads.

No FastAPI starts.

No Redis connects.

Nothing.


---

Layer 6

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8001"]

This does not run during build either.

It only runs when:

docker run ...

is executed.


---

Your answer

> Only copy...file will get reexecute



Almost.

More precisely:

✅ During COPY, files are only copied.

❌ Python files are NOT executed.

Later, when container starts:

docker run recommender-api

then:

app.py executes
↓
startup event runs
↓
model loads
↓
similarity loads
↓
FastAPI starts


---

Now the important Docker interview concept

Suppose your image is already built.

Then you change only:

print("Hello")

inside app.py.

What layers changed?

FROM python:3.11         ← reused
WORKDIR /app             ← reused
COPY requirements.txt    ← reused
RUN pip install          ← reused
COPY . .                 ← rebuilt

Only the last layer rebuilds.

Docker reuses everything above it.

This is called Layer Caching.

That's why Docker builds are fast after the first build.


---

Quick check:

If you add a new package to requirements.txt, which layers will Docker rebuild?

COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .

Think through it and tell me which layers are reused and which are rebuilt. This is the key concept before we write the actual Dockerfile for Day 15. 🐳

Perfect. 🎉

The important line is:

naming to docker.io/library/recommendation-api:latest

and there are no ERROR or FAILED messages.

That means:

✅ Dockerfile is valid
✅ Image built successfully
✅ Dependencies installed inside image
✅ Project copied into image
✅ Docker image created


---

What just happened behind the scenes?

Before today:

Your Laptop
 ├── Python
 ├── FastAPI
 ├── Redis package
 ├── Pandas
 ├── Sklearn
 └── Project

Your project depended on your machine.


---

After build:

Docker Image
 ├── Python
 ├── FastAPI
 ├── Redis package
 ├── Pandas
 ├── Sklearn
 └── Your Project

Everything is packed into one portable artifact.

You can now give that image to another machine and it can run without manually installing Python packages.

This is one of the biggest industry uses of Docker.


---

Quick Concept Check

Suppose tomorrow your laptop crashes.

You buy a new laptop.

Without Docker:

Install Python
Install FastAPI
Install Pandas
Install Redis package
Install Surprise
Install Sklearn
Fix version conflicts
Pray

With Docker:

docker pull recommendation-api
docker run recommendation-api

Everything already exists inside the image.


---

Next Step: Run the Container

Execute:

docker run -p 8001:8001 recommendation-api

Meaning

Container Port : 8001
      ↓
Host Port      : 8001

So:

localhost:8001

on your laptop reaches:

FastAPI inside container


---

Expected Logs

You should see something like:

Loaded: model/svd_model_v1.pkl
Loaded: model/similarity_v1.pkl
Resource loaded...
Uvicorn running on http://0.0.0.0:8001

Notice:

0.0.0.0

instead of

127.0.0.1

This is important inside containers.


---

After running:

Open:

http://localhost:8001/docs

and test:

/recommend/1

Tell me:

1. Does the container start?


2. Do the models load?


3. Does /docs open?



Then we'll move to the next big concept:

Why Redis will fail inside Docker right now and how Docker networking solves it 🚀🐳

That's the point where Docker Compose starts making sense.

Excellent. 🎯

This is actually the exact problem I wanted you to hit before introducing Docker networking.

Your container is running:

FastAPI ✅
Model Loading ✅
Similarity Matrix ✅
Redis ❌

The key error:

Error 111 connecting to localhost:6379. Connection refused.


---

Why does this happen?

Before Docker:

Windows Machine
│
├── FastAPI
└── Redis

Your code:

redis.Redis(
    host="localhost",
    port=6379
)

worked because:

localhost
=
your Windows machine


---

After Docker

Now your FastAPI is inside a container:

Windows
│
├── Redis
│
└── Docker Container
     └── FastAPI

When FastAPI says:

host="localhost"

it means:

Look inside THIS container

not

Look at my Windows laptop

Inside the container:

Container
│
├── FastAPI
└── No Redis

So Docker tries:

localhost:6379

inside the container.

Finds nothing.

Returns:

Connection Refused


---

Real-world analogy

Think of your container as an apartment.

Before Docker:

You lived in a house.

Kitchen
Bedroom
Redis
FastAPI

Everyone could see everyone.


---

After Docker:

Apartment A
└── FastAPI

Apartment B
└── Redis

FastAPI says:

Go to localhost

It looks around Apartment A.

Redis isn't there.

So it fails.


---

This is exactly why Docker Compose exists

Soon we will create:

docker-compose.yml

which will start:

Container 1 -> FastAPI
Container 2 -> Redis

and automatically create a private network:

FastAPI ---> redis

Then code becomes:

redis.Redis(
    host="redis",
    port=6379
)

Notice:

localhost ❌
redis     ✅

because Docker Compose creates a DNS name called:

redis

for the Redis container.


---

For now

This error is expected.

Your Docker image is working correctly.

The failure proves:

✅ FastAPI container started

✅ Models loaded

✅ Startup event executed

✅ API reachable

✅ Docker image valid

❌ Redis networking not configured yet


---

Quick Concept Question

Suppose we create:

services:
  api:
    ...

  redis:
    ...

inside docker-compose.yml.

Inside the API container, which host should we use?

A)

host="localhost"

B)

host="redis"

And explain why in one sentence before we build the Compose file. This is the most important Docker networking concept of the whole day.

No problem. This is one of those concepts that usually clicks after seeing the failure happen, which you've now seen. 😄

Let's use your current setup.


---

Right now

You have:

Windows Machine
│
├── Redis running on localhost:6379
│
└── Docker Container
    └── FastAPI

Inside your code:

redis.Redis(
    host="localhost",
    port=6379
)


---

What does localhost mean?

Many beginners think:

localhost = my laptop

But actually:

localhost = THIS machine/process environment

Whoever says "localhost" means themselves.


---

Example 1: Before Docker

FastAPI runs directly on Windows.

Windows
│
├── FastAPI
└── Redis

FastAPI says:

host="localhost"

Meaning:

Connect to Redis on THIS Windows machine

Works ✅


---

Example 2: Inside Docker

Now FastAPI runs inside a container.

Container
└── FastAPI

FastAPI says:

host="localhost"

Meaning:

Connect to Redis on THIS container

Docker checks:

Container
│
└── FastAPI

No Redis exists there.

Result:

Connection refused


---

Now imagine Docker Compose

Compose creates:

Docker Network
│
├── api container
│   └── FastAPI
│
└── redis container
    └── Redis

Compose automatically creates names.

api
redis

like DNS names.


---

FastAPI can now say:

host="redis"

Meaning:

Find the container named redis

Docker networking does:

api container
      |
      v
redis container

and the connection succeeds.


---

Real-world analogy

Think of an apartment building.

localhost

Means:

My own apartment

So if Dad is in Apartment A and says:

Bring me the TV from localhost

he means:

Bring me the TV from Apartment A

not from the whole building.


---

Docker Compose service name

Now Dad says:

Go to Apartment B

That's like:

host="redis"

He is explicitly referring to another apartment.


---

Therefore

When we create:

services:
  api:
    ...

  redis:
    ...

the API container should use:

redis.Redis(
    host="redis",
    port=6379
)

not

host="localhost"

because Redis lives in a different container.


---

Interview-style summary

Why doesn't localhost work inside Docker Compose?

Because localhost refers to the current container itself. Redis runs in a separate container, so the API must connect using the Redis service name (redis) over Docker's internal network.


---

If this makes sense, next we'll create the docker-compose.yml file and you'll see both containers (FastAPI + Redis) start together with a single command:

docker compose up

This is where Docker starts feeling really powerful. 🚀🐳

Awesome. 🚀

You've now reached the point where most beginners start understanding why Docker Compose exists instead of just memorizing commands.


---

Day 15 - Part 4 : Docker Compose

Current situation:

Windows
│
├── Redis
│
└── Docker Container
     └── FastAPI

Problem:

host="localhost"

fails because FastAPI is inside the container.


---

What we want

Docker Network
│
├── API Container
│     └── FastAPI
│
└── Redis Container
      └── Redis

And Docker automatically connects them.


---

Step 1

Create a file:

docker-compose.yml

in project root.

Same level as:

Dockerfile
requirements.txt
api/
src/
model/


---

Step 2

Paste:

version: "3.9"

services:

  redis:
    image: redis:7-alpine
    container_name: redis-server
    ports:
      - "6379:6379"

  api:
    build: .
    container_name: recommendation-api

    ports:
      - "8001:8001"

    depends_on:
      - redis

Save.


---

Let's understand every line


---

Service 1

redis:

Create a Redis container.


---

Image

image: redis:7-alpine

Docker Hub already contains Redis.

No installation needed.

Think:

docker pull redis

automatically.


---

Container Name

container_name: redis-server

Human-friendly name.

Instead of:

f8d4d6aab23

you'll see:

redis-server


---

Ports

ports:
  - "6379:6379"

Meaning:

Laptop Port 6379
        ↓
Container Port 6379


---

Service 2

api:

Your recommendation system.


---

Build

build: .

Means:

Look for Dockerfile
Build image
Start container


---

Container Name

container_name: recommendation-api

Easy identification.


---

Port Mapping

ports:
  - "8001:8001"

Meaning:

localhost:8001
      ↓
FastAPI container


---

depends_on

depends_on:
  - redis

Means:

Start Redis first
Then API

Not a guarantee Redis is fully ready, but good enough for now.


---

Step 3 (Important)

Now modify:

redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

to:

redis_client = redis.Redis(
    host="redis",
    port=6379,
    decode_responses=True
)

Notice:

localhost ❌
redis ✅

Why?

Because Compose creates a network.

Inside that network:

redis

becomes a DNS name.

Your API container can locate Redis by name.


---

Step 4

Stop any running containers.

Check:

docker ps

If anything running:

docker stop <container_id>


---

Step 5

From project root:

docker compose up

Expected:

Creating redis-server
Creating recommendation-api

Redis ready
FastAPI ready
Loaded: model/svd_model_v1.pkl
Loaded: model/similarity_v1.pkl
Resource loaded...


---

Before running

Quick question:

Suppose we execute:

docker compose up

How many containers should Docker start?

And what are their names?

Answer that first. If that's clear, then run it and show me the output. 🐳🚀

Exactly. 🎯

Docker Compose should create:

Container 1
└── redis-server

Container 2
└── recommendation-api

And Docker automatically creates an internal network:

recommendation-api
        │
        ▼
    redis-server

So now when your code says:

host="redis"

Docker resolves:

redis
   ↓
redis container IP

without you needing to know any IP addresses.


---

One tiny correction

Technically the service name is:

redis:

while the container name is:

container_name: redis-server

For Docker networking, we usually use:

host="redis"

because Docker Compose guarantees the service name DNS entry.


---

Before you run

Make sure you've changed:

host="localhost"

to

host="redis"

in your FastAPI code.

Otherwise the same Redis connection error will happen.


---

Then run:

docker compose up

and watch the logs.

You should see both containers starting.

After that:

Open:

http://localhost:8001/docs

and test:

/recommend/1

If everything works, you've just built your first multi-container ML application:

FastAPI Container
        │
        ▼
Redis Container

which is much closer to how production systems are deployed.

Run it and show me the output/logs. If there are any errors, we'll debug them together. 🚀🐳

Yes, that's completely okay 👍

In fact, what you're mentioning is a different deployment path than Docker Compose.

There are actually two parallel concepts:


---

Path 1: Docker Image Distribution (Docker Hub)

This is what you're describing:

docker build -t recommendation-api .

↓

docker login

↓

docker tag recommendation-api yourusername/recommendation-api:latest

↓

docker push yourusername/recommendation-api:latest

Now anyone can do:

docker pull yourusername/recommendation-api:latest

and run your API.

Think:

GitHub      -> Source Code
Docker Hub  -> Built Application


---

Path 2: Multi-Container Deployment (Docker Compose)

This is where we currently are.

Problem:

FastAPI needs Redis

So instead of manually starting:

docker run redis

and

docker run recommendation-api

we use:

docker-compose.yml

and start both together:

docker compose up


---

Industry Workflow

Usually:

Development

Docker Compose
├── API
├── Redis
├── Database
└── Monitoring

because developers need all services.


---

Deployment

Build image:

docker build -t recommendation-api .

Push:

docker push username/recommendation-api

Deploy server pulls image:

docker pull username/recommendation-api


---

For Our Project

My recommended order:

Today

✅ Dockerfile

✅ Docker Image Build

✅ Docker Compose

✅ FastAPI + Redis containers communicating


---

Day 16 (Hardening)

.dockerignore

Environment variables

Cleanup

Better logging

Production configuration



---

Day 17 (GitHub Ready)

README

Architecture diagram

Screenshots

Resume bullet points



---

Day 18–19 (Redis Completion)

Redis persistence concepts

Distributed caching concepts

Multi-worker concepts

Production discussion



---

Day 20

Interview preparation

Resume optimization

End-to-end project review



---

So you are not missing anything by not pushing yet.

Actually, I would wait until:

Dockerfile
+
Docker Compose
+
Redis

are fully working.

Then we'll push a polished image to Docker Hub once, rather than pushing a partially complete version multiple times.

So the next step is still:

host="redis"


docker compose up

Let's get the multi-container setup working first, then we'll do Docker Hub publishing. That mirrors how teams typically work in real projects. 🚀

Also, one quick check: did you already change

host="localhost"

to

host="redis"

before running Compose? That's the most common thing people forget at this step.

First, your code looks fine for Compose networking.

redis_client = redis.Redis(
    host="redis",
    port=6379,
    decode_responses=True
)

Inside Docker Compose:

FastAPI Container
      |
      | asks for host="redis"
      |
      v
Redis Container

Docker automatically resolves the service name redis to the Redis container IP.


---

About that earlier error

The error:

{
  "status": "FAILED",
  "error": "invalid literal for int() with base 10: 'redis'"
}

is not coming from the code you pasted.

I suspect one of these:

Check Dockerfile

If you have:

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "redis"]

❌ Wrong

Should be:

CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8001"]


---

Check docker-compose.yml

If you added:

environment:
  PORT: redis

❌ Wrong

Should be:

environment:
  PORT: 8001


---

One thing missing

In your compose file, add:

api:
  build: .
  container_name: recommendation-api

  ports:
    - "8001:8001"

  depends_on:
    - redis

  restart: always

Not mandatory, but useful.


---

Now your main question

> Suppose changes done what commands should I rerun?



This is a very important Docker workflow concept.


---

Case 1: Python code changed

Example:

app.py
hybrid.py
content_based.py

changed.

Rebuild image:

docker compose up --build

or

docker compose up --build -d

Docker will:

Reuse cached layers
Re-copy source code
Create new image
Start containers


---

Case 2: requirements.txt changed

Example:

redis
fastapi
pandas

new package added.

Need rebuild:

docker compose up --build

because pip install layer changed.


---

Case 3: docker-compose.yml changed

Example:

ports:
  - "8002:8001"

changed.

Run:

docker compose down
docker compose up --build


---

Case 4: Dockerfile changed

Example:

RUN apt-get install ...

changed.

Run:

docker compose down
docker compose up --build


---

Case 5: Just restart containers

No code changes.

docker compose restart


---

Case 6: Everything broken, start fresh

docker compose down

docker compose build --no-cache

docker compose up

This forces Docker to rebuild every layer from scratch.


---

Commands you'll use 90% of the time

During development:

docker compose up --build

Check running containers:

docker ps

Stop everything:

docker compose down

View logs:

docker compose logs

View API logs:

docker compose logs api

View Redis logs:

docker compose logs redis


---

Quick check before we continue:

Run

docker ps

and tell me:

1. Is recommendation-api running?


2. Is redis-server running?



If both are running, we'll do the final verification that FastAPI is actually reading/writing cache through Redis inside Docker. 🚀